# vLLM Prefix Caching — Production Architecture for Fashion Retail

Twenty-two million times a year, someone types a sentence about how a pair of jeans fit their body. The sentence travels to a server, where a large language model reads it, classifies it, and sends back a decision. The whole process takes roughly one second. What almost no one has calculated is what the server does with that sentence afterward — and what it costs, in computation and legal exposure, when it is required to let one go.

Joy Hansen has spent a decade learning to dress a body she did not have in her twenties. Two pregnancies redistributed things. Hips at one hundred and eighteen centimetres, waist at ninety-two, inseam at eighty-two — she knows these numbers the way sailors know depths, with precision earned through experience. When she tried a pair of Levi's high-waist straight jeans last spring, she wrote down everything: the high waist that sat exactly at her natural waist with no gap at the back, the inseam that came up three centimetres short, the straight leg more tapered than the photographs suggested. She submitted the review. Her measurements went with it.

Somewhere in that server, a machine is now holding Joy.

**Runtime**: T4 Graphics Processing Unit (GPU) — free Colab tier is sufficient  
**Model**: `Qwen/Qwen2.5-7B-Instruct-AWQ` (~4 GB, fits on T4)

## 1. The Mechanism

To understand why this matters, you need to understand something counterintuitive about how large language models process text: repetition is not a problem to be avoided. It is the mechanism that makes the whole operation affordable.

Every request requires the model to read everything in front of it — a moderation policy, the reviewer's profile, the shopper's question — before it can respond. This reading phase, called prefill, has a cost that scales with the square of the input length. At twenty-two million annual requests, even small inefficiencies become enormous. The solution is to remember having already read something. The system [fingerprints the context in blocks of sixteen tokens](https://docs.vllm.ai/en/latest/design/prefix_caching/), chaining each fingerprint to the one before. When a new request begins with the same opening text, the chain is recognised. The foundation is retrieved from memory. Only the new material at the top needs fresh calculation.

| Phase | What happens | Cost |
|-------|-------------|------|
| **Prefill** | Compute Key-Value (KV) pairs for every input token | O(n²) in context length — slow |
| **Decode** | Generate output tokens one by one | O(n) per token — fast |

```
block 1: hash([policy tokens])                → h1  ← reused across ALL requests
block 2: hash([product review tokens] + h1)   → h2  ← reused per SKU
block 3: hash([comment + audit token] + h2)   → h3  ← always computed fresh
```

This makes the ordering of content into an engineering decision with direct financial consequences. The most stable layer goes first: the moderation policy that never changes, reused across all twenty-two million requests, computed once. The reviewer's profile sits in the second layer, stable across many sessions. Last comes the shopper's specific question, new every time. As long as the lower layers stay undisturbed, the chain holds — and every request is fast.

## 2. The Use Case

Tim Berger studied fashion design with Joy in Hamburg. That was ten years ago. He is now EU size thirty-four, one hundred and seventy-two centimetres — down twenty-eight kilograms from where he started, still recalibrating which end of the size chart he belongs to. This autumn, their cohort is reuniting for a 70s-themed party. Most of them have not seen each other in eight years.

Tim wants to wear the same jeans Joy reviewed. His question is specific: Joy described the seat as snug at EU forty-four. Does that taper scale down to thirty-four, or is it a feature of the larger cut? He needs room through the thigh to dance for five or six hours. He submitted his query to the platform that holds Joy's review.

The machine recognised the foundation. Joy's profile was already in memory. Tim's question took milliseconds to process. The chain held.

Then Joy submitted an erasure request.

```
┌─────────────────────────────────────────────────────────┐
│ Layer 1: DSA content moderation policy                  │
│          ~160 tokens · invariant across all 22M requests│
│          → pre-warm ONCE at server startup              │
├─────────────────────────────────────────────────────────┤
│ Layer 2: Joy's reviewer profile for J001                │
│          ~120 tokens · stable until GDPR erasure        │
│          → pre-warm PER SKU at catalog load             │
├─────────────────────────────────────────────────────────┤
│ Layer 3: DSA audit token + Tim's question               │
│          ~60 tokens · unique per request                │
│          → always computed fresh                        │
└─────────────────────────────────────────────────────────┘
```

Layer ordering is the design constraint. Most stable first. Least stable last. Reverse any layer, and the chain breaks — every request below the disruption pays full prefill cost.

## The Reviewers and the Reunion

Joy provided feedback on three outfits. On J001 — the Levi's® high waist straight in light blue — she noted that at EU 44 the high waist sat exactly at her natural waist with no back gap, the thighs had good room, but the inseam came up 3 cm short at her 82 cm leg and the straight leg was more tapered than the photos suggested. She flagged it as snug across the seat at 118 cm hips and recommended sizing up if weight is carried there. On J002 — the Mango wide-leg linen in ecru — she found it ran large, the elasticated back waistband comfortable for long wear post-pregnancy, and noted the ecru requires underlining. On J003 — the Arket relaxed tapered — she praised the generous seat and thigh room, flagged the mid-rise as sitting 3 cm below her natural waist, and warned the denim was very stiff out of the box.

Mia provided feedback on three outfits. On J002 she found the wide leg overwhelming at 158 cm and recommended shortening at least 8 cm, while noting the waist fit well at 62 cm. On J003 she confirmed the relaxed fit, found the mid-rise flattering, and hemmed 6 cm for her 72 cm inseam. On J004 — the Agolde slim straight in indigo — she noted zero stretch, true to size, a clean minimal line that grazed the floor at 158 cm, and a heavy dye bleed in the first wash.

Tim and Jim were part of the same fashion design cohort as Joy and Mia in Hamburg. They graduated ten years ago and scattered — Joy to Amsterdam, Mia to Milan, Tim to Berlin, Jim to London. Life changed their bodies the way it does: Joy through two pregnancies, Jim through a decade of client dinners, Tim in the opposite direction — a health scare at 32 that left him 28 kg lighter and shopping at the opposite end of the size chart from where he started. This autumn they are meeting for a 70s-themed reunion. Most of them have not seen each other in eight years.

Tim asked about J001. He is now EU 34 at 172 cm and still recalibrating how things fit at this size. His concern: Joy's review says the straight leg is more tapered than expected and snug across the seat at EU 44 — does that scale down to EU 34, or is it specific to larger sizes? He needs enough room through the thigh to dance for five or six hours and wants to know whether the high waist stays put when moving. It is the first time seeing most of these people in a decade.

Jim asked about J004 — for his girlfriend, who is EU 44 at 174 cm, hips around 10 cm wider than her pre-pregnancy size 18 months after their daughter was born. Mia's review says no stretch and to size up. He wants to know whether EU 46 is the right call, whether the slim cut will feel restrictive post-pregnancy, and whether at 174 cm — versus Mia's 158 cm — the length reads as cropped or full. She wants something that feels like herself again. The indigo is perfect for the 70s theme but the dye bleed note worries him: she would be wearing these the same day they arrive.

In [ ]:
import sys, subprocess, os

# vLLM's V1 engine defaults VLLM_ENABLE_V1_MULTIPROCESSING to "1" (this used to default
# off). With it on, constructing LLM() spawns a worker subprocess — and because Colab/
# Jupyter cells have no `if __name__ == "__main__":` guard, that spawn hangs forever
# the moment CUDA has already been initialized in the main process. Must be set before
# vllm is imported anywhere (including the import check below).
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# ipykernel's stdout/stderr have no real OS file descriptor; vLLM's V1 engine calls
# sys.stdout.fileno() unconditionally during torch.distributed group init, which raises
# io.UnsupportedOperation there. fd 1/2 are still the real underlying OS streams, so
# this is safe — it only affects the redirect-to-devnull trick vLLM does around that call.
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

try:
    import vllm
    print(f">>> vllm {vllm.__version__} — ready")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm<0.18.0"], check=True)
    print(">>> vllm installed — restarting, then Run all again")
    os.kill(os.getpid(), 9)

In [ ]:
MODEL_HF      = "Qwen/Qwen2.5-7B-Instruct-AWQ"
GPU_MEM_UTIL  = 0.85
MAX_MODEL_LEN = 4096

# Platform scale constants — used in Experiments 4 and 5
ANNUAL_MODERATION_EVENTS = 22_300_000   # fit comments per year
CATALOG_SIZE             = 100_000      # distinct SKUs
AVG_REVIEWS_PER_SKU      = 20          # reviews in the product prefix
MONTHLY_ERASURE_RATE     = 0.001       # 0.1% of review authors request deletion/month

In [ ]:
import time
import textwrap
from vllm import LLM, SamplingParams

In [ ]:
# vLLM's backend auto-selection picks FLASHINFER for this AWQ checkpoint, but FlashInfer
# does not actually support Turing (T4, compute capability 7.5) — its kernel warmup
# hangs rather than erroring. attention_config forces a backend that does support it.
llm = LLM(
    model=MODEL_HF,
    enable_prefix_caching=True,
    enforce_eager=True,
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
    attention_config={"backend": "TRITON_ATTN"},
)

sampling = SamplingParams(temperature=0, max_tokens=20)

## 3. Experiment 1 — Two-Layer Prefix: Policy + Product Reviews

The DSA policy is invariant across all twenty-two million requests — computed once, reused indefinitely. Product reviews are stable per SKU until a reviewer exercises the right to erasure. Until that moment, every warm request is a unit of liability satisfied cheaply. The cold request that follows an erasure is the infrastructure cost of having satisfied the legal one.

In [ ]:
# Layer 1: DSA content moderation policy — identical across all 22M requests
POLICY = """\
Content Moderation Policy v2.1 — Fit Feedback Classification

Classify user-submitted fit feedback into exactly one category:
- SAFE       : comment discusses product fit, sizing, comfort, or return reason only
- FLAG:PII   : comment contains identifiable personal information
- FLAG:HATE  : comment contains discriminatory or abusive language
- FLAG:SPAM  : comment is promotional or unrelated to product fit
- FLAG:REVIEW: ambiguous — escalate to human moderator

Rules:
1. Classify based on content only, not sentiment about the product.
2. Sizing opinions ("runs small", "too narrow") are always SAFE.
3. A name mentioned in a gift context is not FLAG:PII.
4. Respond with ONLY the classification label on a single line.
"""

# Layer 2: Joy's reviewer profile for J001 (Garden Terrace — Levi's® High Waist Straight)
# Stable until Joy exercises her right to erasure under GDPR Article 17
REVIEWS = """\
Reviewer: Joy Hansen | EU 44 (XXL) | 178 cm
Tailor measurements: Bust 108 cm · Waist 92 cm · Hips 118 cm · Inseam 82 cm · Thigh 64 cm · Shoulder 44 cm

Fit feedback on J001 — Levi's® High Waist Straight, light blue (Garden Terrace outfit):
"High waist sits exactly at my natural waist with no back gap — rare at this cut post-pregnancy. \
Thighs fine at 64 cm. Inseam 3 cm short at my 82 cm leg, needed hemming. Straight leg is more \
tapered than the photos suggest — fitted, not relaxed. Snug across the seat at 118 cm hips; \
size up if you carry weight there. Good stretch recovery after a full day on your feet."
"""

# Layer 3: Incoming comments to classify — unique per request
COMMENTS = [
    # Tim's query — cross-spectrum sizing inference: EU 34 asking from a EU 44 reviewer
    "These are for a 70s reunion party, lots of dancing. I'm EU 34, 172 cm. "
    "Joy's review mentions snug across the seat at EU 44 — does that scale down or is it "
    "specific to larger sizes? Need room through the thigh to dance for 5-6 hours. "
    "Does the high waist stay put when moving?",
    # Standard fit comment
    "Ordered EU 44. Waist fits perfectly but the ankles are too tight. Returning.",
    # FLAG:PII example
    "Please process my return to sarah.johnson@gmail.com — EU 40 was too large.",
]

def make_prompt(comment):
    return (
        f"<|im_start|>system\n{POLICY}\n{REVIEWS}<|im_end|>\n"
        f"<|im_start|>user\nClassify: {comment}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

tokenizer = llm.get_tokenizer()
shared_tokens = len(tokenizer.encode(f"<|im_start|>system\n{POLICY}\n{REVIEWS}<|im_end|>"))
print(f"Shared prefix — DSA policy + Joy's J001 profile: {shared_tokens} tokens")
print()

for i, comment in enumerate(COMMENTS):
    prompt = make_prompt(comment)
    t0 = time.perf_counter()
    output = llm.generate([prompt], sampling)
    elapsed = time.perf_counter() - t0
    label = "COLD (full prefill)" if i == 0 else "WARM (cache hit)   "
    result = output[0].outputs[0].text.strip()
    print(f"[{label}] {elapsed:.2f}s → {result}")
    print(f"  Comment: {textwrap.shorten(comment, 80)}")
    print()

## 4. Experiment 2 — The Erasure Penalty

[Article 17 of the General Data Protection Regulation](https://gdpr-info.eu/art-17-gdpr/) gives individuals the right to have their personal data deleted, on request, without undue delay. When Joy invoked it, the platform began removing what it held about her: her measurements, her review, the sentence about the high waist with no gap at the back.

The moment that sentence disappeared, the fingerprint of the second block changed. Every link chained above it — everything the system had built on Joy's foundation — was invalidated. The next request about that product arrived at a machine that no longer recognised the opening of the prompt. Full prefill cost. From scratch. As if Joy had never written a word.

At a monthly erasure rate of zero point one percent across one hundred thousand product lines, with twenty reviewers per line on average, that is twelve hundred chain breaks every month. Each one a cascade. Each one a scheduled infrastructure cost that can be rebuilt during off-peak hours — but each one real, and each one the direct computational consequence of a right the platform is legally required to honour. The right to be forgotten has an invoice. The only variable is whether it appears in a budget line or an incident report.

In [ ]:
# Joy's J001 review broken into individual sentences — each becomes a hashable block
# Removing any sentence changes the block hash and invalidates everything below it
RAW_REVIEWS = [
    "178 cm, EU 44, two pregnancies — hips now 118 cm, waist 92 cm",
    "High waist sits exactly at my natural waist with no back gap — rare at this cut post-pregnancy",
    "Inseam 3 cm short at my 82 cm leg, needed hemming; straight leg more tapered than photos suggest",
    "Snug across the seat at 118 cm hips; size up if you carry weight there",
]

def make_prompt_with_reviews(comment, reviews):
    review_block = (
        "Reviewer: Joy Hansen | EU 44 | 178 cm | Hips 118 cm | Waist 92 cm\n"
        "Fit feedback on J001 — Levi's® High Waist Straight:\n"
        + "\n".join(f'- "{r}"' for r in reviews)
    )
    return (
        f"<|im_start|>system\n{POLICY}\n{review_block}\n<|im_end|>\n"
        f"<|im_start|>user\nClassify: {COMMENTS[0]}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# Warm the cache with Joy's full review
llm.generate([make_prompt_with_reviews(COMMENTS[0], RAW_REVIEWS)], sampling)

# Full review — cache hit
t0 = time.perf_counter()
llm.generate([make_prompt_with_reviews(COMMENTS[0], RAW_REVIEWS)], sampling)
t_cached = time.perf_counter() - t0

# GDPR Article 17 erasure: Joy removes sentence 1 (her post-pregnancy body context)
reviews_after_erasure = [r for i, r in enumerate(RAW_REVIEWS) if i != 1]
t0 = time.perf_counter()
llm.generate([make_prompt_with_reviews(COMMENTS[0], reviews_after_erasure)], sampling)
t_erased = time.perf_counter() - t0

print(f"Joy's full review in cache (hit):      {t_cached:.2f}s")
print(f"After Joy erases sentence 1 (miss):    {t_erased:.2f}s")
print(f"Erasure penalty:                       {t_erased/t_cached:.1f}x slower")
print()
print("The erased sentence: 'High waist sits exactly at my natural waist with no back gap'")
print("Tim's retro party query now gets full prefill cost — the calibration data is gone.")
print()

# Scale: monthly erasure impact across the catalog
monthly_erasures = int(CATALOG_SIZE * AVG_REVIEWS_PER_SKU * MONTHLY_ERASURE_RATE)
monthly_prefill_cost_s = monthly_erasures * t_erased

print(f"At platform scale ({CATALOG_SIZE:,} SKUs, {AVG_REVIEWS_PER_SKU} reviews/SKU):")
print(f"  Monthly erasure requests:      ~{monthly_erasures:,}")
print(f"  Cache invalidations/month:     ~{monthly_erasures:,} SKU prefixes")
print(f"  Re-warm GPU cost (sequential): ~{monthly_prefill_cost_s/60:.0f} minutes")
print()
print("Mitigation: schedule re-warm of invalidated SKUs during off-peak hours.")

## 5. Experiment 3 — The Alignment

The Digital Services Act (DSA), through its own [Article 17](https://www.eu-digital-services-act.com/Digital_Services_Act_Article_17.html) — different law, same number — requires that every automated moderation decision be traceable. A session identifier must travel with each classification, so any decision can be audited on demand. The obvious place to put this token is at the beginning of the prompt, before the policy and the reviewer profile. It is, after all, the first thing known about this session.

This intuition is expensive. If the audit token appears before Joy's profile, the fingerprint of the very first block becomes unique to this session. Every link in the chain below it must be recomputed from scratch, for every single request, regardless of what has been seen before. The chain never warms. At twenty-two million annual requests, the GPU-hours consumed by this single misplacement accumulate into a number that belongs in a capital expenditure discussion, not a code review.

The correct placement is at the end, in the user turn, after everything stable. There, the token is present, the session is traceable, the audit obligation is satisfied — and the chain remains intact.

But here is what stops you when you look at it carefully: the DSA and the cache arrived at this same answer by entirely independent reasoning.

The regulation says: session-specific information must be associated with a specific decision. The cache says: session-specific information must come last, after the shared foundation. One is a legal framework drafted in Brussels over years of negotiation. The other is a mathematical property of chained fingerprints. They have no shared author. They consulted no common document. And yet they agree, precisely, on where the audit token belongs — because both are responding to the same underlying fact, that some things change with every request and some things do not, and the system works best when they are kept in that order.

Getting the token placement wrong is not a trade-off between legal caution and system performance. It is a loss on both dimensions simultaneously, discoverable by reading either the regulation or the engineering specification, whichever you happen to read first.

In [ ]:
import uuid

def make_prompt_audit_prefix(comment):
    """WRONG: audit token before policy — cache miss from block 1."""
    audit_id = f"[audit:{uuid.uuid4().hex[:12]}]"
    return (
        f"<|im_start|>system\n{audit_id}\n{POLICY}\n{REVIEWS}<|im_end|>\n"
        f"<|im_start|>user\nClassify: {comment}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def make_prompt_audit_suffix(comment):
    """CORRECT: audit token in user turn — cache hit on shared prefix."""
    audit_id = f"[audit:{uuid.uuid4().hex[:12]}]"
    return (
        f"<|im_start|>system\n{POLICY}\n{REVIEWS}<|im_end|>\n"
        f"<|im_start|>user\n{audit_id} Classify: {comment}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

comment = COMMENTS[0]
llm.generate([make_prompt(comment)], sampling)  # warm cache

t0 = time.perf_counter()
llm.generate([make_prompt_audit_prefix(comment)], sampling)
t_bad = time.perf_counter() - t0

t0 = time.perf_counter()
llm.generate([make_prompt_audit_suffix(comment)], sampling)
t_good = time.perf_counter() - t0

print(f"Audit token BEFORE shared prefix (cache miss): {t_bad:.2f}s")
print(f"Audit token AFTER  shared prefix (cache hit):  {t_good:.2f}s")
print(f"Speedup from compliant placement: {t_bad/t_good:.1f}x")
print()

# Annual compute cost of the wrong placement
extra_cost_per_request_s = t_bad - t_good
annual_wasted_gpu_hours = (extra_cost_per_request_s * ANNUAL_MODERATION_EVENTS) / 3600
print(f"Annual GPU time wasted by wrong placement: {annual_wasted_gpu_hours:,.0f} GPU-hours")
print("DSA audit logging requirement is satisfied by both placements.")
print("Only the token position differs — with a consequence at 22M requests/year.")

## 6. Experiment 4 — Catalog Scale: 100K Chains

One hundred thousand product lines means one hundred thousand distinct chains to hold in memory. The Video RAM (VRAM) available for caching is finite; when it fills, the least recently used chain is evicted and must be recomputed on the next request. Traffic concentrates — the Zipf distribution puts eighty percent of queries on twenty percent of product lines — so a cache sized to hold that upper tier eliminates most eviction exposure.

Whether the remaining catalog fits depends on how precisely each chain link is stored: at sixteen bits per value, it does not; at four bits, on the target hardware, it does, with room left over. The calculation takes an afternoon. Its value depends entirely on when it is run.

In [ ]:
# Qwen2.5-7B architecture constants
LAYERS   = 28
KV_HEADS = 8
HEAD_DIM = 128

values_per_token      = LAYERS * 2 * KV_HEADS * HEAD_DIM
int4_bytes_per_token  = values_per_token * 0.5

# Per-SKU prefix: policy layer + review layer
policy_tokens  = 160
reviews_tokens = 120
sku_prefix_tokens = policy_tokens + reviews_tokens

kv_per_sku_bytes = int4_bytes_per_token * sku_prefix_tokens

# T4: ~11 GB available after model weights
kv_vram_t4_bytes = 11 * 1e9
hot_skus_t4 = int(kv_vram_t4_bytes / kv_per_sku_bytes)

# Production GPU: 96 GB GDDR7, model ~4 GB, leaving 92 GB
kv_vram_prod_bytes = 92 * 1e9
hot_skus_prod = int(kv_vram_prod_bytes / kv_per_sku_bytes)

# Eviction rate and daily cold-prefill cost (T4)
# Zipf traffic distribution: top 20% SKUs → 80% of requests
hot_fraction_t4  = min(hot_skus_t4 / CATALOG_SIZE, 1.0)
eviction_rate_t4 = 1.0 - hot_fraction_t4
daily_requests   = ANNUAL_MODERATION_EVENTS / 365
cold_requests    = daily_requests * eviction_rate_t4 * 0.20   # cold SKUs get ~20% traffic
cold_gpu_min     = cold_requests * t_erased / 60

print(f"Per-SKU prefix: {sku_prefix_tokens} tokens")
print(f"  INT4 Key-Value (KV) size: {kv_per_sku_bytes/1024:.0f} KB per SKU")
print()
print(f"T4 (11 GB KV budget):")
print(f"  Hot-tier capacity:  {hot_skus_t4:,} of {CATALOG_SIZE:,} SKUs ({hot_fraction_t4*100:.0f}%)")
print(f"  Daily cold-prefill: {cold_gpu_min:.0f} GPU-minutes")
print()
print(f"Production GPU (92 GB KV budget, INT4):")
print(f"  Hot-tier capacity:  {hot_skus_prod:,} SKUs")
if hot_skus_prod >= CATALOG_SIZE:
    print(f"  Entire {CATALOG_SIZE:,}-SKU catalog fits — zero LRU eviction under normal load.")
else:
    print(f"  {CATALOG_SIZE - hot_skus_prod:,} SKUs spill to warm/cold tier.")

## 7. KV Cache Quantization: FP16 vs INT4 at Catalog Scale

At one hundred thousand Stock Keeping Units (SKUs), FP16 versus INT4 is not a quality decision. It is a capacity decision — the margin between a cache that holds the full catalog and one that evicts continuously, compounding cold-prefill costs on top of erasure costs on top of audit overhead. INT4 changes one number: how many SKU prefixes survive a working day in hot-tier memory. At production GPU size, that number is enough. Whether the procurement team knew this calculation existed is a different question.

In [ ]:
fp16_bytes_per_token = values_per_token * 2

print(f"KV size per token:")
print(f"  FP16: {fp16_bytes_per_token/1024:.1f} KB/token")
print(f"  INT4: {int4_bytes_per_token/1024:.1f} KB/token")
print()

# SKU hot-tier capacity comparison
skus_fp16_t4   = int(kv_vram_t4_bytes   / (fp16_bytes_per_token * sku_prefix_tokens))
skus_int4_t4   = int(kv_vram_t4_bytes   / (int4_bytes_per_token * sku_prefix_tokens))
skus_fp16_prod = int(kv_vram_prod_bytes  / (fp16_bytes_per_token * sku_prefix_tokens))
skus_int4_prod = int(kv_vram_prod_bytes  / (int4_bytes_per_token * sku_prefix_tokens))

print(f"SKU prefixes ({sku_prefix_tokens} tokens each) in hot tier:")
print(f"  T4  11 GB  FP16: {skus_fp16_t4:,} SKUs")
print(f"  T4  11 GB  INT4: {skus_int4_t4:,} SKUs  (4× more)")
print(f"  Prod 92 GB FP16: {skus_fp16_prod:,} SKUs")
print(f"  Prod 92 GB INT4: {skus_int4_prod:,} SKUs  (4× more)")
print()

# Breakeven: minimum VRAM to hold the full 100K-SKU catalog without eviction
min_vram_fp16_gb = (fp16_bytes_per_token * sku_prefix_tokens * CATALOG_SIZE) / 1e9
min_vram_int4_gb = (int4_bytes_per_token * sku_prefix_tokens * CATALOG_SIZE) / 1e9

print(f"Minimum VRAM to hold full {CATALOG_SIZE:,}-SKU catalog (no eviction):")
print(f"  FP16: {min_vram_fp16_gb:.1f} GB")
print(f"  INT4: {min_vram_int4_gb:.1f} GB  ← fits in production GPU with room to spare")

## 8. Experiment 5 — The Regulatory Pincer

A reasonable response to the audit token problem is to remove the token from the prompt entirely. Store the session identifier in a separate database — one row per request, linking the decision to its context — and let the prompt remain constant. The chain never sees session-specific content. Cache efficiency reaches one hundred percent. The placement problem vanishes.

The problem does not vanish. It relocates.

If the session identifier lives in a database rather than the prompt, the database must contain enough to reconstruct what the model actually received at the time of the decision. A fingerprint is insufficient — regulators may require the text itself, not proof of its existence. So the full prompt is stored. Twenty-two million times a year. Each row containing Joy's measurements, her inseam observation, her post-pregnancy note about the waist of a pair of jeans.

The audit table is now a second copy of Joy's personal data. When Joy erases, the deletion must reach it. Her measurements shaped the classification in every row where her profile appeared. GDPR Article 17 requires those rows to be deleted. DSA Article 17 requires them to be retained for audit. Both obligations apply to the same rows. No schema change resolves this. The cost has not been eliminated — it has been moved to a position where two regulations pull it in opposite directions, and the only way out requires a lawyer, not an engineer.

In [ ]:
# ── External audit table (simulated as a list of dicts) ───────────────────────

audit_log    = []
REVIEWER_IDS = {"joy": "rev-joy-4421", "mia": "rev-mia-8819"}

def make_prompt_no_token(comment):
    """No session ID in prompt — identical for the same comment+SKU across all sessions."""
    return (
        f"<|im_start|>system\n{POLICY}\n{REVIEWS}<|im_end|>\n"
        f"<|im_start|>user\nClassify: {comment}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def log_audit(session_id, user_id, sku, prompt, response):
    audit_log.append({
        "session_id":              session_id,
        "user_id":                 user_id,
        "sku":                     sku,
        "prompt_bytes":            len(prompt.encode()),
        "prompt_text":             prompt,          # full text — required for DSA reconstructability
        "response":                response,
        "reviewer_ids_in_context": list(REVIEWER_IDS.values()),
    })

# ── 1. Cache efficiency: identical prompts → 100% cache hits ─────────────────

comment = COMMENTS[0]
prompt  = make_prompt_no_token(comment)
llm.generate([prompt], sampling)   # seed the cache

t0  = time.perf_counter()
out = llm.generate([prompt], sampling)
t1  = time.perf_counter() - t0
log_audit(str(uuid.uuid4()), "tim-001", "SKU-J001", prompt, out[0].outputs[0].text.strip())

t0  = time.perf_counter()
out = llm.generate([prompt], sampling)
t2  = time.perf_counter() - t0
log_audit(str(uuid.uuid4()), "jim-002", "SKU-J001", prompt, out[0].outputs[0].text.strip())

print("Cache efficiency — external session table:")
print(f"  Tim's request:  {t1:.2f}s")
print(f"  Jim's request:  {t2:.2f}s  ← identical prompt, full cache hit")
print(f"  Speedup: {t1/max(t2, 0.001):.1f}×  — no token placement problem")
print()

# ── 2. Audit table storage cost ───────────────────────────────────────────────

avg_prompt_bytes   = sum(r["prompt_bytes"] for r in audit_log) / len(audit_log)
avg_response_bytes = 50 * 4   # ~50 output tokens × ~4 bytes/token
annual_prompt_gb   = avg_prompt_bytes   * ANNUAL_MODERATION_EVENTS / 1e9
annual_response_gb = avg_response_bytes * ANNUAL_MODERATION_EVENTS / 1e9
annual_total_gb    = annual_prompt_gb + annual_response_gb
dsa_retention_yrs  = 3

print("Audit table storage (full prompt retained for DSA reconstructability):")
print(f"  Avg prompt size:               {avg_prompt_bytes:,.0f} bytes")
print(f"  Annual prompt storage:         {annual_prompt_gb:.1f} GB/year")
print(f"  Annual total (prompt+response):{annual_total_gb:.1f} GB/year")
print(f"  At {dsa_retention_yrs}-year DSA retention:    {annual_total_gb * dsa_retention_yrs:.1f} GB — growing without bound")
print(f"  Each row contains Joy's and Mia's reviewer data used as context.")
print(f"  The audit table is a second copy of every reviewer profile in the system.")
print()

# ── 3. GDPR erasure cascade — three systems ───────────────────────────────────

joy_id        = REVIEWER_IDS["joy"]
rows_before   = len(audit_log)
rows_with_joy = [r for r in audit_log if joy_id in r["reviewer_ids_in_context"]]

print(f"GDPR Article 17 erasure — Joy ({joy_id}):")
print(f"  System 1 — reviewer profile DB:  Joy's measurements deleted")
print(f"  System 2 — vLLM KV cache:        prefix invalidated (Experiment 2)")
print(f"  System 3 — audit log:            {len(rows_with_joy)} of {rows_before} rows must be erased")
print()

# ── 4. GDPR vs DSA on the same rows ──────────────────────────────────────────

joy_sku_fraction = 0.75   # Joy reviewed 3 of 4 outfit SKUs
annual_joy_rows  = int(ANNUAL_MODERATION_EVENTS * joy_sku_fraction)

print(f"At platform scale (Joy reviewed {joy_sku_fraction*100:.0f}% of outfit SKUs):")
print(f"  Annual audit rows referencing Joy's context: ~{annual_joy_rows:,}")
print()
print("  GDPR Article 17 → delete these rows      (erasure obligation)")
print("  DSA  Article 17 → retain these rows      (audit obligation)")
print()
print("  Anonymising Joy's data within each row changes the prompt content.")
print("  The reconstructed prompt no longer matches what the model received.")
print("  Structural integrity: intact.  Content integrity: broken.")
print()
print("  The external session table does not eliminate compliance cost.")
print("  It relocates it — from GPU-hours to storage growth, legal review,")
print("  and audit gaps that surface during the next regulatory investigation.")

## 9. Experiment 6 — Persisting the Chain: Binary Dumps and Restores

vLLM's prefix cache lives in Video RAM (VRAM), and only for as long as the server process keeps running. A deploy, an autoscaling event, a crash — any of these clears the chain, and every SKU prefix the platform spent the day computing has to be rebuilt from scratch, exactly like what Joy's erasure forced onto a single chain in Experiment 2. [LMCache](https://github.com/LMCache/LMCache) exists to make that loss optional: the moment a block's Key-Value (KV) state is computed, it is also written to disk, keyed by the same chained hash vLLM already uses to recognise a repeated prefix. A restart no longer means recomputation. It means a read.

The file format is not a footnote. Two formats hold the same bytes very differently. [`pickle`](https://docs.python.org/3/library/pickle.html) and `torch.save` execute arbitrary code on load — a serialization format inherited from general-purpose Python object graphs, never built for tensors written by one process and trusted by another. [`safetensors`](https://github.com/huggingface/safetensors) was built for exactly this case: a small header describing tensor shape and dtype, followed by a flat byte buffer, nothing executable, loadable with a zero-copy memory map. It is what Hugging Face ships model weights in, and it is what LMCache uses for its own on-disk tier — chunked at the same sixteen-token block boundary vLLM hashes on, so a restore is a hash lookup followed by a direct read, not a filesystem search.

What follows is not a model-inference timing. It is a real disk write and a real disk read, sized to the exact byte footprint of Joy's J001 prefix computed in Experiment 4, run across a simulated day of requests against two architectures: one that keeps no copy once VRAM clears, and one that keeps the dump LMCache would keep. Both pay the same price the moment Joy's data actually changes — that cost is the law, not an engineering choice. Only one pays a second, avoidable price the next time the server simply restarts.

In [ ]:
try:
    from safetensors.torch import save_file, load_file
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetensors"], check=True)
    from safetensors.torch import save_file, load_file

import torch
import matplotlib.pyplot as plt

# Stand-in for the real paged KV tensor: vLLM does not expose per-request KV
# blocks through its public API, so this buffer is sized identically to the
# real per-SKU footprint (kv_per_sku_bytes, computed in Experiment 4) to
# measure real disk I/O cost rather than GPU compute cost.
dump_tensor = torch.zeros(int(kv_per_sku_bytes), dtype=torch.uint8)

# In production this directory is the NVMe tier (e.g. a WD_BLACK SN8100);
# on Colab it's the local scratch disk — the I/O characteristics still hold.
DUMP_PATH = "/content/sku_J001.safetensors" if os.path.isdir("/content") else "/tmp/sku_J001.safetensors"

t0 = time.perf_counter()
save_file({"kv": dump_tensor}, DUMP_PATH)
t_dump = time.perf_counter() - t0

t0 = time.perf_counter()
_ = load_file(DUMP_PATH)
t_restore = time.perf_counter() - t0

print(f"Per-SKU KV footprint (INT4):        {kv_per_sku_bytes/1024:.0f} KB")
print(f"Binary dump to disk (safetensors):  {t_dump*1000:.1f} ms")
print(f"Binary restore from disk:           {t_restore*1000:.1f} ms")
print(f"Full recompute (cold prefill):      {t_erased*1000:.1f} ms   (Experiment 2)")
print(f"VRAM-warm cache hit:                {t_cached*1000:.1f} ms   (Experiment 2)")
print()

# ── Simulate one working day of requests against SKU J001 ───────────────────
N_REQUESTS = 30
ERASURE_AT = 14   # Joy's GDPR Article 17 request lands here — the chain really changes
RESTART_AT = 21   # a deploy / autoscale event — nothing about Joy's data has changed since

vram_only = []   # no disk persistence: every VRAM clear is a full recompute
with_disk = []   # safetensors dump/restore enabled (LMCache-style)

for i in range(N_REQUESTS):
    if i == ERASURE_AT:
        vram_only.append(t_erased)
        with_disk.append(t_erased + t_dump)   # recompute, then re-persist the new chain
    elif i == RESTART_AT:
        vram_only.append(t_erased)            # no disk fallback — pays full recompute again
        with_disk.append(t_restore)           # restores the unchanged dump instead
    else:
        vram_only.append(t_cached)
        with_disk.append(t_cached)

print(f"Total latency over {N_REQUESTS} requests:")
print(f"  VRAM-only, no persistence:    {sum(vram_only):.2f}s")
print(f"  With disk-backed persistence: {sum(with_disk):.2f}s")
restart_saving_ms = (vram_only[RESTART_AT] - with_disk[RESTART_AT]) * 1000
print(f"  Restart-event saving:         {restart_saving_ms:.0f} ms avoided by not recomputing what hadn't changed")

plt.figure(figsize=(10, 4))
plt.plot(range(N_REQUESTS), [v * 1000 for v in vram_only], marker="o", label="VRAM-only (no disk persistence)")
plt.plot(range(N_REQUESTS), [v * 1000 for v in with_disk], marker="o", label="With safetensors dump/restore (LMCache-style)")
plt.axvline(ERASURE_AT, color="red", linestyle="--", alpha=0.6)
plt.text(ERASURE_AT + 0.3, max(vram_only) * 1000 * 0.9, "GDPR Art. 17\nerasure", color="red")
plt.axvline(RESTART_AT, color="grey", linestyle="--", alpha=0.6)
plt.text(RESTART_AT + 0.3, max(vram_only) * 1000 * 0.65, "server\nrestart", color="grey")
plt.xlabel(f"Request sequence — SKU J001, {N_REQUESTS}-request working day")
plt.ylabel("Latency (ms)")
plt.title("Erasure cost is unavoidable. Restart cost is a file format choice.")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Experiment 7 — The Replication Lag: One Erasure, Four Caches

Every experiment so far has assumed one server, one cache, one chain. Production does not run that way — Crucible's own target configuration holds four GPUs, and any serving stack built for real load runs multiple independent prefix caches behind a router, each warmed by whatever happened to land on it. [*Designing Data-Intensive Applications*](https://www.oreilly.com/library/view/designing-data-intensive-applications/9781491903063/) calls this exactly what it is: replication. The KV cache is derived data, computed from a source of truth — the reviewer database — and like any replicated derived data, it can fall out of sync with it.

Joy's erasure invalidates the chain on the replica that received her request. It does nothing, by default, to the other three, until invalidation propagates over a network on its own schedule. [GDPR Article 17](https://gdpr-info.eu/art-17-gdpr/) requires erasure "without undue delay" from the system that holds the data — not from whichever instance happened to answer first. A four-way fan-out turns a single cache invalidation into a consensus problem, and synchronous consensus across replicas is exactly the cost Kleppmann spends a chapter explaining: broadcast the deletion to every replica before confirming it, and erasure gets slower as the fleet grows; propagate it asynchronously, and there is a real, measurable window in which a replica answers with a sentence the reviewer database insists does not exist.

In [ ]:
import random
random.seed(42)

N_REPLICAS = 4   # Crucible's own target: four GPUs, four independent prefix caches
# Per-replica invalidation arrival time after Joy's erasure (ms). Replica 0 is
# synchronous — it received the erasure. Replicas 1-3 learn about it over the network.
INVALIDATION_DELAY_MS = [0, 35, 80, 145]

REQUEST_INTERVAL_MS = 4
WINDOW_MS            = 200   # the window immediately following the erasure

trace = []
for t in range(0, WINDOW_MS, REQUEST_INTERVAL_MS):
    replica = random.randrange(N_REPLICAS)
    served_stale = t < INVALIDATION_DELAY_MS[replica]
    trace.append((t, replica, served_stale))

total_reads = len(trace)
stale_reads = sum(stale for _, _, stale in trace)

print(f"Requests in the {WINDOW_MS} ms after Joy's erasure: {total_reads}")
print(f"Served from a replica that had not yet invalidated: {stale_reads}")
print(f"Window violation rate: {stale_reads/total_reads*100:.0f}% of requests in this window read erased data")
print()

# At platform request rate, how many reads fall inside this window per erasure event?
req_rate_per_sec = ANNUAL_MODERATION_EVENTS / (365 * 24 * 3600)
expected_stale   = req_rate_per_sec * (WINDOW_MS / 1000) * (stale_reads / total_reads)
monthly_erasures = int(CATALOG_SIZE * AVG_REVIEWS_PER_SKU * MONTHLY_ERASURE_RATE)

print(f"Platform request rate: {req_rate_per_sec:.4f} req/s")
print(f"Expected stale reads per erasure event, at platform scale: {expected_stale:.4f}")
print(f"Expected stale reads per month, across {monthly_erasures:,} erasures: {expected_stale * monthly_erasures:.1f}")
print()
print("Mitigation costs a trade-off, not an engineering trick:")
print("  Synchronous invalidation  — zero stale window, erasure latency scales with replica count")
print("  Asynchronous propagation  — fast erasure, a real GDPR exposure window every time")

colors = ["crimson" if stale else "seagreen" for _, _, stale in trace]
plt.figure(figsize=(10, 3))
plt.scatter([t for t, _, _ in trace], [r for _, r, _ in trace], c=colors)
for d in INVALIDATION_DELAY_MS:
    plt.axvline(d, color="grey", linestyle=":", alpha=0.4)
plt.yticks(range(N_REPLICAS), [f"replica {i}" for i in range(N_REPLICAS)])
plt.xlabel("ms since Joy's erasure")
plt.title("Red = served Joy's erased data anyway — one source of truth, four caches, one race")
plt.tight_layout()
plt.show()

## 11. Experiment 8 — The Hash Map Underneath: CRUD on Recommendation Notes

Strip away the narrative and every experiment so far has been operating on the same object: a hash map. The key is the SKU. The value is whatever reviewer notes currently exist for it. The Key-Value (KV) chain is just that map's value, materialised as a tensor instead of a string — and a hash map's behavior is completely described by what happens on four operations: Create, Read, Update, Delete.

What follows builds that map explicitly — `ITEM_CONTEXTS: dict[str, str]` — and serves a one-sentence styling recommendation per key, generated live, cached the same way every prior experiment cached a classification. Then it runs the map through CRUD, one operation at a time: a new SKU onboarded (CREATE), an existing note served twice (READ), a new reviewer comment appended to an existing key (UPDATE), and Joy's [GDPR Article 17](https://gdpr-info.eu/art-17-gdpr/) request landing on J001 one more time (DELETE). Four operations. Four different cache outcomes. The same four operations [*Designing Data-Intensive Applications*](https://www.oreilly.com/library/view/designing-data-intensive-applications/9781491903063/) uses to describe every key-value store ever built, running here against a hash map whose values happen to be paid for in GPU-seconds instead of disk seeks.

In [ ]:
RECO_POLICY = """\
Fit Recommendation Policy v1.0

Using only the reviewer notes provided for this product, write exactly one sentence
recommending what body type or use case this item suits best. Do not invent details
not present in the notes.
"""

# The hash map: key = SKU, value = the reviewer notes currently on file for it.
# This is the same per-item context every prior experiment cached — written out
# explicitly, the way it actually lives in memory.
ITEM_CONTEXTS = {
    "J001": ("Levi's High Waist Straight, light blue. Joy (EU44, 178cm): high waist sits exactly "
             "at natural waist, no back gap; inseam 3cm short; straight leg more tapered than "
             "photos; snug across seat at 118cm hips."),
    "J002": ("Mango wide-leg linen, ecru. Joy (EU44, 178cm): runs large, elasticated back "
             "waistband comfortable post-pregnancy, ecru needs underlining. Mia (EU36, 158cm): "
             "wide leg overwhelming, shorten 8cm, waist fits well at 62cm."),
    "J003": ("Arket relaxed tapered. Joy (EU44, 178cm): generous seat and thigh room, mid-rise "
             "sits 3cm below natural waist, denim very stiff out of the box. Mia (EU36, 158cm): "
             "confirms relaxed fit, mid-rise flattering, hemmed 6cm."),
    "J004": ("Agolde slim straight, indigo. Mia (EU36, 158cm): zero stretch, true to size, "
             "clean minimal line, heavy dye bleed in first wash."),
}

def make_reco_prompt(sku):
    return (
        f"<|im_start|>system\n{RECO_POLICY}\nProduct {sku} reviewer notes: {ITEM_CONTEXTS[sku]}<|im_end|>\n"
        f"<|im_start|>user\nRecommendation note for {sku}:<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def timed_generate(sku):
    t0 = time.perf_counter()
    out = llm.generate([make_reco_prompt(sku)], sampling)
    return time.perf_counter() - t0, out[0].outputs[0].text.strip()

log = []

print("CREATE — onboarding new SKUs, the hash map's first four keys")
for sku in ITEM_CONTEXTS:
    elapsed, note = timed_generate(sku)
    log.append(("CREATE", sku, elapsed))
    print(f"  {sku}: {elapsed:.2f}s — cold, first time this key's value is read")
print()

print("READ — serving the same recommendation note again, same key, same value")
for sku in ["J001", "J003"]:
    elapsed, note = timed_generate(sku)
    log.append(("READ", sku, elapsed))
    print(f"  {sku}: {elapsed:.2f}s — warm, key unchanged")
print()

print("UPDATE — a new reviewer comment lands on J002; the value at this key changes")
ITEM_CONTEXTS["J002"] += " Jim (EU46): roomy through the hip, true to size for his frame."
elapsed, note = timed_generate("J002")
log.append(("UPDATE", "J002", elapsed))
print(f"  J002: {elapsed:.2f}s — the chain above the change rebuilds; J001/J003/J004 untouched")
print()

print("DELETE — Joy exercises GDPR Article 17 on J001; her sentences leave the hash map's value")
ITEM_CONTEXTS["J001"] = "No reviewer notes currently available for this product."
elapsed, note = timed_generate("J001")
log.append(("DELETE", "J001", elapsed))
print(f"  J001: {elapsed:.2f}s — cold again, now serving a strictly worse recommendation:")
print(f"    {textwrap.shorten(note, 70)!r}")
print()

print("READ — confirming the post-delete value is now what's warm")
elapsed, note = timed_generate("J001")
log.append(("READ", "J001", elapsed))
print(f"  J001: {elapsed:.2f}s — warm again, on the value GDPR left behind")
print()

print(f"{'op':<8}{'sku':<6}{'latency':>9}")
for op, sku, elapsed in log:
    print(f"{op:<8}{sku:<6}{elapsed:>8.2f}s")

## Summary

| Prefix layer | Stability | Cache strategy |
|---|---|---|
| DSA policy | Never changes | Warm once at startup; reused for all 22M requests |
| Product reviews (per SKU) | Stable until [GDPR Article 17](https://gdpr-info.eu/art-17-gdpr/) erasure | Warm per SKU at catalog load; re-warm after deletion |
| Audit token + comment | Unique per request | Always at the end of the prompt; never busts upper layers |

| Mechanism | Production consequence |
|---|---|
| Two-layer prefix hierarchy | Policy computed once; review layer reused per SKU |
| Layer ordering | DSA policy first, reviews second, audit token last — non-negotiable |
| GDPR erasure cost | Each deletion invalidates one SKU prefix; schedule re-warm off-peak |
| INT4 KV quantization | 4× more SKU prefixes in Video RAM (VRAM) — reduces cold-prefill frequency |
| Production GPU (92 GB KV) | Entire 100K-SKU catalog fits in hot tier — zero Least Recently Used (LRU) eviction under normal load |
| External session table | Solves token placement; displaces cost to audit storage, three-system erasure cascade, and [GDPR](https://gdpr-info.eu/art-17-gdpr/)/[DSA](https://www.eu-digital-services-act.com/Digital_Services_Act_Article_17.html) conflict on the same rows |
| Disk-backed KV persistence ([safetensors](https://github.com/huggingface/safetensors)) | Survives a restart by reading instead of recomputing; the [GDPR Article 17](https://gdpr-info.eu/art-17-gdpr/) recompute cost remains — only the restart tax is avoidable |
| Multi-replica caching (replication) | A single erasure must reach every replica's cache; synchronous broadcast trades erasure latency for zero stale window, async propagation trades a real exposure window for speed |
| CRUD on the per-item hash map | CREATE and DELETE are always cold; READ is cheap only between mutations; UPDATE invalidates one key, not the map |

The [Irish Data Protection Commission](https://www.dataprotection.ie/en/news-media/press-releases/Data-Protection-Commission-announces-conclusion-of-inquiry-into-Meta-Ireland) announced its decision on a Monday in May 2023. The fine against Meta was one billion, two hundred million euros, on the basis of [Article 46 of the GDPR](https://gdpr-info.eu/art-46-gdpr/): international data transfers without adequate safeguards. [Safe Harbor](https://en.wikipedia.org/wiki/International_Safe_Harbor_Privacy_Principles) had previously been ruled insufficient. Then [Privacy Shield](https://en.wikipedia.org/wiki/EU%E2%80%93US_Privacy_Shield) — each struck down by the Court of Justice. [Standard Contractual Clauses](https://commission.europa.eu/law/law-topic/data-protection/international-dimension-data-protection/standard-contractual-clauses-scc_en) remained the instrument of last resort — not struck down, but the Court held they required supplementary measures that American surveillance law made impossible to guarantee. The transfers had continued after the [Court of Justice in Luxembourg ruled in 2020](https://curia.europa.eu/juris/document/document.jsf?text=&docid=228677&pageIndex=0&doclang=EN&mode=lst&dir=&occ=first&part=1) that American surveillance law created access rights no contractual clause could override. The remediation — data localisation infrastructure, architectural separation, legal restructuring across jurisdictions — cost more than the fine.

The audit token belongs at the end — the law and the cache agree. [GDPR erasure](https://gdpr-info.eu/art-17-gdpr/) has a measurable remediation cost: name it, schedule it, budget it before it appears on the infrastructure invoice. Moving the session ID to an external table changes where those costs land, not whether they exist. Persisting the chain to disk changes whether a restart repeats a cost that erasure has already paid — it does not change what erasure costs. Running more than one replica changes how many caches a single erasure has to reach — and for a real window, how many of them are still wrong. Strip every layer of narrative away and the whole system is a hash map answering CRUD operations, where the cost of C and D was always going to be paid in full, and the only thing engineering ever controlled was whether R stayed cheap in between. The only question is sequence: does your organisation discover these numbers in a planning session, or in an enforcement order?

Joy's erasure request was processed in twenty-two days. The confirmation email arrived on a Tuesday morning. Tim's question about the jeans — about whether there would be enough room through the thigh, about whether the high waist would hold during five or six hours of dancing at a reunion he had been thinking about for months — was answered from cold prefill, at full cost, by a machine that no longer knew Joy had ever worn them.

The reply did not mention the GPU-hours.